# DCCP pinned CPU validation
Run the next cell in Colab or Jupyter with Python 3.10+. It creates a fresh isolated environment, verifies an exact source commit, installs pinned optional adapters, executes tests and the real-count fixture checks, and downloads results. No GPU is needed.
The report preserves OOD misses. Passing software checks does not establish empirical biological validation.


In [ ]:
from pathlib import Path
import datetime
import json
import os
import subprocess
import sys
import tempfile
import zipfile

REPO_REF = "3ed75b8153ae36dc0716d26f81ee4a8cf8d3786d"
workspace = Path(tempfile.mkdtemp(prefix="dccp-cpu-"))
repo = workspace / "repo"
venv = workspace / "venv"
subprocess.run(["git", "clone", "https://github.com/Virelion-Biotech/Virelion-DCCP.git", str(repo)], check=True)
subprocess.run(["git", "checkout", "--detach", REPO_REF], cwd=repo, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
assert actual == REPO_REF
subprocess.run([sys.executable, "-m", "venv", str(venv)], check=True)
python = venv / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
subprocess.run([str(python), "-m", "pip", "install", ".[test,validation,cardivex,cardisim]"], cwd=repo, check=True)
log_path = workspace / "pytest.log"
with log_path.open("w") as log:
    test = subprocess.run([str(python), "-m", "pytest", "-q", "--cov=dccp", "--cov-branch"], cwd=repo, stdout=log, stderr=subprocess.STDOUT)
print(log_path.read_text())
test.check_returncode()
subprocess.run([str(python), "scripts/validate_cpu.py"], cwd=repo, check=True)
report_path = repo / "validation/cpu/results.json"
report = json.loads(report_path.read_text())
assert report["passed"], report["checks"]
print("Empirical status:", report["empirical_validation"])
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output = Path.cwd() / ("DCCP_CPU_Validation_" + stamp + ".zip")
with zipfile.ZipFile(output, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(report_path, "results.json")
    archive.write(repo / "validation/data/GSE240848_host_subset.json", "GSE240848_host_subset.json")
    archive.write(repo / "docs/CPU_AUDIT.md", "CPU_AUDIT.md")
    archive.write(log_path, "pytest.log")
    archive.writestr("commit.txt", actual + "\n")
print("Saved:", output)
try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(str(output))
